# Real-time Data Streaming with Kafka + KsqlDB

**DADS6005 Quiz 2 — Food Survey Analytics Pipeline**

## Pipeline Architecture

```
CSV (food_coded.csv)
    │
    ▼  (Python Producer)
Kafka Topic
    │
    ├──▶ KsqlDB STREAM (Real-time processing)
    ├──▶ KsqlDB TABLE  (Materialized aggregations)
    └──▶ Python Consumer (Read results)
```

## Dataset: Food Survey

ชุดข้อมูลพฤติกรรมการกินของนักศึกษา 125 คน (University of Ottawa)
ประกอบด้วย 62 คอลัมน์ เช่น GPA, เพศ, calories, comfort food, อาหารไทย ฯลฯ

---

## 1. Setup & Imports

In [ ]:
import json
import time
import csv
import pandas as pd
import requests
from kafka import KafkaProducer, KafkaConsumer
from kafka.admin import KafkaAdminClient, NewTopic

## 2. Helper Functions

In [ ]:
KAFKA_BROKER = 'localhost:29092'
KSQLDB_URL = 'http://localhost:8088'

def ksqldb_request(payload):
    """Send request to KsqlDB REST API"""
    resp = requests.post(f'{KSQLDB_URL}/ksql', json=payload)
    if resp.status_code == 200:
        return resp.json()
    print(f'[ERROR] {resp.status_code}: {resp.text}')
    return None

def ksqldb_query(sql):
    """Execute KsqlDB DDL/DML"""
    return ksqldb_request({'ksql': sql, 'streamsProperties': {}})

def ksqldb_pull(sql):
    """Pull query (return rows immediately)"""
    result = ksqldb_request({'ksql': sql, 'streamsProperties': {}})
    if result:
        return result[0].get('rows', [])
    return []

def list_topics():
    consumer = KafkaConsumer(bootstrap_servers=KAFKA_BROKER)
    topics = consumer.topics()
    consumer.close()
    return sorted(topics)

def create_topic(topic_name, partitions=1):
    admin = KafkaAdminClient(bootstrap_servers=KAFKA_BROKER)
    try:
        admin.create_topics([NewTopic(topic_name, partitions, 1)])
        return f'[OK] Created topic: {topic_name}'
    except Exception as e:
        return f'[!] {e}'

## 3. Check Services

In [ ]:
print("=== Services Check ===")

try:
    r = requests.get(f'{KSQLDB_URL}/info', timeout=5)
    print(f'[OK] KsqlDB Server: {r.json().get("KsqlServerInfo", {}).get("kafkaClusterId", "running")}')
except Exception as e:
    print(f'[FAIL] KsqlDB Server: {e}')

try:
    topics = list_topics()
    print(f'[OK] Kafka Broker: {len(topics)} topics found')
    for t in topics:
        print(f'      - {t}')
except Exception as e:
    print(f'[FAIL] Kafka Broker: {e}')

---
## 4. Stream CSV Data to Kafka

ส่งข้อมูล food_coded.csv ไปยัง Kafka topic `food_coded_json` แบบ real-time (1 record/วินาที)

In [ ]:
TOPIC = 'food_coded_json'
print(create_topic(TOPIC))

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

MAX_ROWS = 10  # จำกัดเพื่อการสาธิต
with open('food_coded.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        if i >= MAX_ROWS:
            break
        producer.send(TOPIC, value=row)
        print(f'[{i+1}/{MAX_ROWS}] Sent user_id={row["ids"]}')
        time.sleep(1)

producer.flush()
print(f'\n[Done] {MAX_ROWS} records sent to "{TOPIC}"')

---
## 5. KsqlDB: Create STREAM

สร้าง STREAM จาก Kafka topic เพื่อให้สามารถ query ด้วย SQL ได้

In [ ]:
ksqldb_query(f'''
    CREATE STREAM IF NOT EXISTS food_stream (
        ids INT,
        gpa VARCHAR,
        gender INT,
        comfort_food VARCHAR,
        comfort_food_reasons VARCHAR,
        calories_day INT,
        fav_cuisine VARCHAR,
        exercise INT,
        veggies_day INT,
        thai_food INT,
        income INT
    ) WITH (
        KAFKA_TOPIC = '{TOPIC}',
        VALUE_FORMAT = 'JSON'
    );
''')
print('[OK] food_stream created')

## 6. Push Query — Real-time Streaming

ดูข้อมูล streaming แบบ real-time (push query) — จะแสดงผลทุกครั้งที่มี record ใหม่

In [ ]:
print("=== Push Query: Real-time food stream ===")
push_sql = "SELECT ids, gpa, comfort_food_reasons, thai_food FROM food_stream WHERE thai_food >= 4 EMIT CHANGES;"

resp = requests.post(
    f'{KSQLDB_URL}/query',
    json={'sql': push_sql, 'streamsProperties': {}},
    stream=True
)

for i, line in enumerate(resp.iter_lines()):
    if line and i < 5:  # แสดง 5 rows แรก
        try:
            data = json.loads(line.decode('utf-8'))
            if 'row' in data:
                print(f'  Row: {data["row"]["columns"]}')
        except:
            pass
    elif i >= 5:
        print('  ... (streaming continues)')
        break

resp.close()

---
## 7. KsqlDB: Materialized Views (Tables)

สร้าง **Materialized Tables** สำหรับ aggregation แบบ real-time

In [ ]:
# Average calories by gender
ksqldb_query('''
    CREATE TABLE IF NOT EXISTS avg_calories_by_gender AS
    SELECT
        CASE WHEN gender = 1 THEN 'Female' ELSE 'Male' END AS gender,
        COUNT(*) AS count,
        ROUND(AVG(calories_day)) AS avg_calories_day
    FROM food_stream
    GROUP BY gender
    EMIT CHANGES;
''')
print('[OK] avg_calories_by_gender table created')

In [ ]:
# Correlation: Exercise frequency vs Diet
ksqldb_query('''
    CREATE TABLE IF NOT EXISTS exercise_vs_diet AS
    SELECT
        exercise,
        COUNT(*) AS count,
        ROUND(AVG(veggies_day), 2) AS avg_veggies,
        ROUND(AVG(calories_day), 2) AS avg_calories,
        ROUND(AVG(gpa), 2) AS avg_gpa
    FROM food_stream
    GROUP BY exercise
    EMIT CHANGES;
''')
print('[OK] exercise_vs_diet table created')

In [ ]:
# Top comfort food reasons
ksqldb_query('''
    CREATE TABLE IF NOT EXISTS top_comfort_reasons AS
    SELECT
        UCASE(comfort_food_reasons) AS reason,
        COUNT(*) AS frequency,
        ROUND(AVG(gpa), 2) AS avg_gpa
    FROM food_stream
    GROUP BY UCASE(comfort_food_reasons)
    EMIT CHANGES;
''')
print('[OK] top_comfort_reasons table created')

In [ ]:
# Favorite cuisines popularity
ksqldb_query('''
    CREATE TABLE IF NOT EXISTS top_cuisines AS
    SELECT
        UCASE(fav_cuisine) AS cuisine,
        COUNT(*) AS popularity
    FROM food_stream
    GROUP BY UCASE(fav_cuisine)
    EMIT CHANGES;
''')
print('[OK] top_cuisines table created')

---
## 8. Pull Queries — Read Aggregated Results

Query ข้อมูลที่ถูก aggregate ไว้ใน Materialized Tables (snapshot)

In [ ]:
print("=== Average Calories by Gender ===")
rows = ksqldb_pull("SELECT * FROM avg_calories_by_gender;")
for r in rows:
    print(f"  {r[0]}: count={r[1]}, avg_calories_day={r[2]}")

In [ ]:
print("=== Exercise vs Diet Correlation ===")
rows = ksqldb_pull("SELECT exercise, count, avg_veggies, avg_calories, avg_gpa FROM exercise_vs_diet ORDER BY exercise;")
for r in rows:
    print(f"  Exercise freq={r[0]}: count={r[1]}, avg_veggies={r[2]}, avg_cal={r[3]}, avg_gpa={r[4]}")

In [ ]:
print("=== Top Comfort Food Reasons ===")
rows = ksqldb_pull("SELECT reason, frequency, avg_gpa FROM top_comfort_reasons ORDER BY frequency DESC LIMIT 5;")
for r in rows:
    print(f"  {r[0]}: {r[1]} times, avg_gpa={r[2]}")

In [ ]:
print("=== Top Favorite Cuisines ===")
rows = ksqldb_pull("SELECT cuisine, popularity FROM top_cuisines ORDER BY popularity DESC LIMIT 5;")
for r in rows:
    print(f"  {r[0]}: {r[1]} votes")

---
## 9. Real-time Consumer

อ่านข้อมูลจาก Kafka topic โดยตรงเพื่อยืนยันว่า stream ทำงาน

In [ ]:
consumer = KafkaConsumer(
    TOPIC,
    auto_offset_reset='earliest',
    bootstrap_servers=KAFKA_BROKER,
    value_deserializer=lambda m: json.loads(m.decode('utf-8'))
)

count = 0
for msg in consumer:
    val = msg.value
    print(f"[#{val['ids']}] GPA={val['gpa']} | comfort_reason={str(val['comfort_food_reasons'])[:30]} | thai_food={val['thai_food']}")
    count += 1
    if count >= 5:
        print("... (5 messages shown)")
        break

consumer.close()

---
## Summary

### ✅ What we did:
1. **Streamed** CSV data → Kafka topic (`food_coded_json`)
2. **Created KsqlDB STREAM** → รองรับ SQL query บน streaming data
3. **Push Queries** → ดูข้อมูลแบบ real-time
4. **Materialized Tables** → Aggregation แบบทันที (avg calories, exercise vs diet, top cuisines)
5. **Pull Queries** → อ่านผลลัพธ์ snapshot

### Key Insights from Food Survey:
- **Stress/Boredom** คือเหตุผลหลักที่คนกิน comfort food
- ความถี่ในการ **ออกกำลังกาย** มีความสัมพันธ์กับ veggies intake
- **Italian cuisine** เป็นที่นิยมสูงสุดในกลุ่มตัวอย่าง
- KsqlDB ทำให้เราสามารถ query ข้อมูล streaming ได้แบบ **real-time** โดยไม่ต้องเขียนโค้ดซับซ้อน

---
*DADS6005 Data Streaming — Quiz 2 (KsqlDB)*